In [1]:
!pip install -q duckdb yfinance pandas numpy plotly pyarrow streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 95.9 MB/s eta 0:00:00


In [2]:
import duckdb
import pandas as pd
import yfinance as yf
from datetime import datetime

DB_PATH = "energy_market.duckdb"

TICKERS = [
    "TAEE11.SA",
    "EGIE3.SA",
    "CEBR3.SA",
    "ENGI11.SA",
    "CMIG4.SA",
    "AXIA3.SA",
    "ALUP11.SA",
    "ENGI11.SA",
    "EQTL3.SA",
    "NEOE3.SA"
]

In [3]:
def create_database():

    conn = duckdb.connect(DB_PATH)

    print("Criando schemas...")

    conn.execute("CREATE SCHEMA IF NOT EXISTS bronze;")
    conn.execute("CREATE SCHEMA IF NOT EXISTS silver;")
    conn.execute("CREATE SCHEMA IF NOT EXISTS gold;")
    conn.execute("CREATE SCHEMA IF NOT EXISTS mart;")

    # =====================================================
    # BRONZE
    # =====================================================

    conn.execute("""
    CREATE TABLE IF NOT EXISTS bronze.stock_fundamentals_raw (

        run_id VARCHAR,

        ticker VARCHAR,

        company_name VARCHAR,

        sector VARCHAR,

        current_price DOUBLE,

        market_cap DOUBLE,

        enterprise_value DOUBLE,

        trailing_pe DOUBLE,

        price_to_book DOUBLE,

        dividend_yield DOUBLE,

        beta DOUBLE,

        ebitda DOUBLE,

        total_debt DOUBLE,

        revenue_growth DOUBLE,

        earnings_growth DOUBLE,

        return_on_equity DOUBLE,

        return_on_assets DOUBLE,

        snapshot_date DATE,

        ingestion_timestamp TIMESTAMP

    )
    """)

    conn.execute("""
    CREATE TABLE IF NOT EXISTS bronze.stock_price_history_raw (

        run_id VARCHAR,

        ticker VARCHAR,

        trade_date DATE,

        open DOUBLE,

        high DOUBLE,

        low DOUBLE,

        close DOUBLE,

        volume BIGINT,

        snapshot_date DATE,

        ingestion_timestamp TIMESTAMP

    )
    """)

    conn.execute("""
    CREATE TABLE IF NOT EXISTS bronze.pipeline_audit (

        run_id VARCHAR,

        start_time TIMESTAMP,

        end_time TIMESTAMP,

        records_loaded INTEGER,

        status VARCHAR
    )
    """)

    conn.execute("""
    CREATE TABLE IF NOT EXISTS bronze.pipeline_logs (

        event_time TIMESTAMP,

        pipeline_step VARCHAR,

        status VARCHAR,

        message VARCHAR
    )
    """)

    # =====================================================
    # SILVER
    # =====================================================

    conn.execute("""
    CREATE TABLE IF NOT EXISTS silver.stock_fundamentals (

        ticker VARCHAR,

        company_name VARCHAR,

        sector VARCHAR,

        current_price DOUBLE,

        market_cap DOUBLE,

        enterprise_value DOUBLE,

        trailing_pe DOUBLE,

        price_to_book DOUBLE,

        dividend_yield DOUBLE,

        beta DOUBLE,

        ebitda DOUBLE,

        total_debt DOUBLE,

        revenue_growth DOUBLE,

        earnings_growth DOUBLE,

        return_on_equity DOUBLE,

        return_on_assets DOUBLE,

        snapshot_date DATE

    )
    """)

    conn.execute("""
    CREATE TABLE IF NOT EXISTS silver.stock_price_history (

        ticker VARCHAR,

        trade_date DATE,

        open DOUBLE,

        high DOUBLE,

        low DOUBLE,

        close DOUBLE,

        volume BIGINT

    )
    """)

    conn.execute("""
    CREATE TABLE IF NOT EXISTS silver.rejected_records (

        ticker VARCHAR,

        error_reason VARCHAR,

        load_timestamp TIMESTAMP
    )
    """)

    conn.execute("""
    CREATE TABLE IF NOT EXISTS silver.data_quality_report (

        execution_time TIMESTAMP,

        rule_name VARCHAR,

        approved INTEGER,

        rejected INTEGER
    )
    """)

    # =====================================================
    # GOLD
    # =====================================================

    conn.execute("""
    CREATE TABLE IF NOT EXISTS gold.valuation_metrics (

        ticker VARCHAR,

        pe_ratio DOUBLE,

        pb_ratio DOUBLE,

        ev_ebitda DOUBLE,

        dividend_yield DOUBLE,

        snapshot_date DATE
    )
    """)

    conn.execute("""
    CREATE TABLE IF NOT EXISTS gold.risk_metrics (

        ticker VARCHAR,

        beta DOUBLE,

        debt_ebitda DOUBLE,

        volatility DOUBLE,

        max_drawdown DOUBLE,

        snapshot_date DATE
    )
    """)

    conn.execute("""
    CREATE TABLE IF NOT EXISTS gold.performance_metrics (

        ticker VARCHAR,

        return_30d DOUBLE,

        return_90d DOUBLE,

        return_180d DOUBLE,

        return_365d DOUBLE,

        snapshot_date DATE
    )
    """)

    # =====================================================
    # MART
    # =====================================================

    conn.execute("""
    CREATE TABLE IF NOT EXISTS mart.dashboard_dataset (

        ticker VARCHAR,

        company_name VARCHAR,

        sector VARCHAR,

        current_price DOUBLE,

        pe_ratio DOUBLE,

        pb_ratio DOUBLE,

        ev_ebitda DOUBLE,

        dividend_yield DOUBLE,

        beta DOUBLE,

        debt_ebitda DOUBLE,

        volatility DOUBLE,

        max_drawdown DOUBLE,

        return_30d DOUBLE,

        return_90d DOUBLE,

        return_180d DOUBLE,

        return_365d DOUBLE,

        snapshot_date DATE
    )
    """)

    conn.close()

    print("Banco criado com sucesso.")

In [4]:
create_database()

Criando schemas...
Banco criado com sucesso.


In [5]:
conn = duckdb.connect(DB_PATH)

tables = conn.sql("""

SELECT
    table_schema,
    table_name

FROM information_schema.tables

ORDER BY table_schema, table_name

""").df()

tables

,table_schema,table_name
0,bronze,pipeline_audit
1,bronze,pipeline_logs
2,bronze,stock_fundamentals_raw
3,bronze,stock_price_history_raw
4,gold,performance_metrics
5,gold,risk_metrics
6,gold,valuation_metrics
7,mart,dashboard_dataset
8,silver,data_quality_report
9,silver,rejected_records


In [6]:
import yfinance as yf
import uuid
from datetime import datetime

In [7]:
def write_log(step, status, message):

    conn = duckdb.connect(DB_PATH)

    conn.execute("""
    INSERT INTO bronze.pipeline_logs
    VALUES (?, ?, ?, ?)
    """, [
        datetime.now(),
        step,
        status,
        message
    ])

    conn.close()

In [8]:
def ingest_fundamentals(run_id):

    conn = duckdb.connect(DB_PATH)

    rows = []

    snapshot_date = datetime.now().date()

    for ticker in TICKERS:

        try:

            print(f"Coletando {ticker}")

            stock = yf.Ticker(ticker)

            info = stock.info

            rows.append({

                "run_id": run_id,

                "ticker": ticker,

                "company_name": info.get("shortName"),

                "sector": info.get("sector"),

                "current_price": info.get("currentPrice"),

                "market_cap": info.get("marketCap"),

                "enterprise_value": info.get("enterpriseValue"),

                "trailing_pe": info.get("trailingPE"),

                "price_to_book": info.get("priceToBook"),

                "dividend_yield": info.get("dividendYield"),

                "beta": info.get("beta"),

                "ebitda": info.get("ebitda"),

                "total_debt": info.get("totalDebt"),

                "revenue_growth": info.get("revenueGrowth"),

                "earnings_growth": info.get("earningsGrowth"),

                "return_on_equity": info.get("returnOnEquity"),

                "return_on_assets": info.get("returnOnAssets"),

                "snapshot_date": snapshot_date,

                "ingestion_timestamp": datetime.now()
            })

            write_log(
                "FUNDAMENTALS",
                "SUCCESS",
                f"{ticker} carregado"
            )

        except Exception as e:

            write_log(
                "FUNDAMENTALS",
                "ERROR",
                f"{ticker} - {str(e)}"
            )

    df = pd.DataFrame(rows)

    conn.register("fundamentals_df", df)

    conn.execute("""
    INSERT INTO bronze.stock_fundamentals_raw
    SELECT *
    FROM fundamentals_df
    """)

    conn.close()

    return len(df)

In [9]:
def ingest_price_history(run_id):

    conn = duckdb.connect(DB_PATH)

    snapshot_date = datetime.now().date()

    all_prices = []

    for ticker in TICKERS:

        try:

            print(f"Histórico {ticker}")

            hist = yf.Ticker(ticker).history(
                period="1y",
                auto_adjust=True
            )

            hist = hist.reset_index()

            hist["run_id"] = run_id

            hist["ticker"] = ticker

            hist["snapshot_date"] = snapshot_date

            hist["ingestion_timestamp"] = datetime.now()

            all_prices.append(hist)

            write_log(
                "PRICE_HISTORY",
                "SUCCESS",
                f"{ticker} histórico carregado"
            )

        except Exception as e:

            write_log(
                "PRICE_HISTORY",
                "ERROR",
                f"{ticker} - {str(e)}"
            )

    prices = pd.concat(all_prices)

    prices = prices.rename(columns={

        "Date":"trade_date",

        "Open":"open",

        "High":"high",

        "Low":"low",

        "Close":"close",

        "Volume":"volume"
    })

    conn.register("price_df", prices)

    conn.execute("""

    INSERT INTO bronze.stock_price_history_raw

    SELECT

        run_id,

        ticker,

        trade_date,

        open,

        high,

        low,

        close,

        volume,

        snapshot_date,

        ingestion_timestamp

    FROM price_df

    """)

    conn.close()

    return len(prices)

In [10]:
def register_audit(
    run_id,
    start_time,
    end_time,
    records_loaded,
    status
):

    conn = duckdb.connect(DB_PATH)

    conn.execute("""

    INSERT INTO bronze.pipeline_audit

    VALUES (?, ?, ?, ?, ?)

    """, [

        run_id,

        start_time,

        end_time,

        records_loaded,

        status

    ])

    conn.close()

In [11]:
def run_bronze_ingestion():

    run_id = str(uuid.uuid4())

    start_time = datetime.now()

    print(f"RUN ID: {run_id}")

    fundamentals_count = ingest_fundamentals(run_id)

    prices_count = ingest_price_history(run_id)

    total_records = (
        fundamentals_count +
        prices_count
    )

    end_time = datetime.now()

    register_audit(
        run_id,
        start_time,
        end_time,
        total_records,
        "SUCCESS"
    )

    print("\nCarga concluída.")
    print(f"Fundamentalistas: {fundamentals_count}")
    print(f"Histórico: {prices_count}")

In [12]:
run_bronze_ingestion()

RUN ID: 99f99c01-db51-4b27-bfc8-fc359bb726f3
Coletando TAEE11.SA
Coletando EGIE3.SA
Coletando CEBR3.SA
Coletando ENGI11.SA
Coletando CMIG4.SA
Coletando AXIA3.SA
Coletando ALUP11.SA
Coletando ENGI11.SA
Coletando EQTL3.SA
Coletando NEOE3.SA
Histórico TAEE11.SA
Histórico EGIE3.SA
Histórico CEBR3.SA
Histórico ENGI11.SA
Histórico CMIG4.SA
Histórico AXIA3.SA
Histórico ALUP11.SA
Histórico ENGI11.SA
Histórico EQTL3.SA
Histórico NEOE3.SA

Carga concluída.
Fundamentalistas: 10
Histórico: 2499


In [13]:
conn = duckdb.connect(DB_PATH)

conn.sql("""
SELECT *
FROM bronze.pipeline_audit
ORDER BY start_time DESC
""").df()

,run_id,start_time,end_time,records_loaded,status
0,99f99c01-db51-4b27-bfc8-fc359bb726f3,2026-06-12 22:23:59.036537,2026-06-12 22:24:02.815639,2509,SUCCESS


In [14]:
conn.sql("""
SELECT *
FROM bronze.pipeline_logs
ORDER BY event_time DESC
LIMIT 50
""").df()

,event_time,pipeline_step,status,message
0,2026-06-12 22:24:02.782823,PRICE_HISTORY,SUCCESS,NEOE3.SA histórico carregado
1,2026-06-12 22:24:02.637625,PRICE_HISTORY,SUCCESS,EQTL3.SA histórico carregado
2,2026-06-12 22:24:02.460917,PRICE_HISTORY,SUCCESS,ENGI11.SA histórico carregado
3,2026-06-12 22:24:02.416712,PRICE_HISTORY,SUCCESS,ALUP11.SA histórico carregado
4,2026-06-12 22:24:02.267206,PRICE_HISTORY,SUCCESS,AXIA3.SA histórico carregado
5,2026-06-12 22:24:02.071379,PRICE_HISTORY,SUCCESS,CMIG4.SA histórico carregado
6,2026-06-12 22:24:01.881936,PRICE_HISTORY,SUCCESS,ENGI11.SA histórico carregado
7,2026-06-12 22:24:01.711094,PRICE_HISTORY,SUCCESS,CEBR3.SA histórico carregado
8,2026-06-12 22:24:01.557029,PRICE_HISTORY,SUCCESS,EGIE3.SA histórico carregado
9,2026-06-12 22:24:01.375699,PRICE_HISTORY,SUCCESS,TAEE11.SA histórico carregado


In [15]:
conn.sql("""
SELECT *
FROM bronze.stock_fundamentals_raw
LIMIT 10
""").df()

,run_id,ticker,company_name,sector,current_price,market_cap,enterprise_value,trailing_pe,price_to_book,dividend_yield,beta,ebitda,total_debt,revenue_growth,earnings_growth,return_on_equity,return_on_assets,snapshot_date,ingestion_timestamp
0,99f99c01-db51-4b27-bfc8-fc359bb726f3,TAEE11.SA,TAESA UNT N2,Utilities,39.69,4.101961e+10,2.392825e+10,39.297028,1.717884,8.2500,-0.014,2.324641e+09,1.185530e+10,0.055,-0.032,0.20559,0.06456,2026-06-12,2026-06-12 22:23:59.462726
1,99f99c01-db51-4b27-bfc8-fc359bb726f3,EGIE3.SA,ENGIE BRASILON NM,Utilities,35.21,4.022034e+10,6.653766e+10,15.648889,2.957084,5.3000,0.302,7.146892e+09,3.242130e+10,0.131,-0.022,0.20204,0.06404,2026-06-12,2026-06-12 22:23:59.672120
2,99f99c01-db51-4b27-bfc8-fc359bb726f3,CEBR3.SA,CEB ON,Utilities,27.02,NaN,1.793019e+09,12.171171,2.059451,14.6000,0.059,1.271710e+08,3.570000e+06,-0.235,-0.376,0.16638,0.05155,2026-06-12,2026-06-12 22:23:59.891844
3,99f99c01-db51-4b27-bfc8-fc359bb726f3,ENGI11.SA,ENERGISA UNT N2,Utilities,47.40,1.451782e+10,5.640542e+10,34.347828,0.922125,6.9500,0.125,8.119768e+09,4.821897e+10,0.070,-0.400,0.12091,0.04653,2026-06-12,2026-06-12 22:24:00.074527
4,99f99c01-db51-4b27-bfc8-fc359bb726f3,CMIG4.SA,CEMIG PN N1,Utilities,10.73,3.069512e+10,4.914859e+10,6.274853,1.062587,11.7400,0.093,7.750017e+09,2.001137e+10,0.063,-0.058,0.17038,0.05927,2026-06-12,2026-06-12 22:24:00.262809
5,99f99c01-db51-4b27-bfc8-fc359bb726f3,AXIA3.SA,AXIA ENERGIAON NM,Utilities,52.00,1.169494e+11,2.050288e+11,11.818182,1.218940,6.9800,0.234,1.337589e+10,7.838196e+10,0.221,NaN,0.07862,0.01973,2026-06-12,2026-06-12 22:24:00.448949
6,99f99c01-db51-4b27-bfc8-fc359bb726f3,ALUP11.SA,ALUPAR UNT N2,Utilities,32.27,1.534942e+10,2.508938e+10,7.733046,1.145382,2.5800,0.175,2.895635e+09,1.457164e+10,0.026,-0.338,0.12679,0.05156,2026-06-12,2026-06-12 22:24:00.628981
7,99f99c01-db51-4b27-bfc8-fc359bb726f3,ENGI11.SA,ENERGISA UNT N2,Utilities,47.40,1.451782e+10,5.640542e+10,34.347828,0.922125,6.9500,0.125,8.119768e+09,4.821897e+10,0.070,-0.400,0.12091,0.04653,2026-06-12,2026-06-12 22:24:00.709871
8,99f99c01-db51-4b27-bfc8-fc359bb726f3,EQTL3.SA,EQUATORIAL ON NM,Utilities,38.77,4.865310e+10,9.872817e+10,41.244682,1.897700,6.1000,0.039,1.188615e+10,5.805280e+10,0.120,-0.240,0.06936,0.04946,2026-06-12,2026-06-12 22:24:00.903490
9,99f99c01-db51-4b27-bfc8-fc359bb726f3,NEOE3.SA,NEOENERGIA ON,Utilities,33.80,4.102635e+10,9.104866e+10,7.716895,1.125504,0.0433,0.284,1.425800e+10,5.901900e+10,0.028,0.731,0.14588,0.06113,2026-06-12,2026-06-12 22:24:01.148521


In [16]:
conn.sql("""
SELECT *

FROM bronze.stock_price_history_raw

ORDER BY trade_date DESC

LIMIT 10
""").df()

,run_id,ticker,trade_date,open,high,low,close,volume,snapshot_date,ingestion_timestamp
0,99f99c01-db51-4b27-bfc8-fc359bb726f3,TAEE11.SA,2026-06-12,39.720001,39.939999,39.430000,39.689999,1026300,2026-06-12,2026-06-12 22:24:01.374883
1,99f99c01-db51-4b27-bfc8-fc359bb726f3,EGIE3.SA,2026-06-12,34.310001,35.939999,34.090000,35.209999,2463200,2026-06-12,2026-06-12 22:24:01.556544
2,99f99c01-db51-4b27-bfc8-fc359bb726f3,CEBR3.SA,2026-06-12,26.709999,27.350000,26.709999,27.020000,700,2026-06-12,2026-06-12 22:24:01.710607
3,99f99c01-db51-4b27-bfc8-fc359bb726f3,ENGI11.SA,2026-06-12,46.560001,47.639999,46.110001,47.400002,1674500,2026-06-12,2026-06-12 22:24:01.881502
4,99f99c01-db51-4b27-bfc8-fc359bb726f3,CMIG4.SA,2026-06-12,10.810000,10.920000,10.720000,10.730000,6539300,2026-06-12,2026-06-12 22:24:02.070927
5,99f99c01-db51-4b27-bfc8-fc359bb726f3,AXIA3.SA,2026-06-12,51.939999,52.840000,51.740002,52.000000,8526200,2026-06-12,2026-06-12 22:24:02.266742
6,99f99c01-db51-4b27-bfc8-fc359bb726f3,ALUP11.SA,2026-06-12,32.270000,32.639999,32.119999,32.270000,562900,2026-06-12,2026-06-12 22:24:02.416220
7,99f99c01-db51-4b27-bfc8-fc359bb726f3,ENGI11.SA,2026-06-12,46.560001,47.639999,46.110001,47.400002,1674500,2026-06-12,2026-06-12 22:24:02.460238
8,99f99c01-db51-4b27-bfc8-fc359bb726f3,EQTL3.SA,2026-06-12,38.720001,39.240002,38.549999,38.770000,9023600,2026-06-12,2026-06-12 22:24:02.636839
9,99f99c01-db51-4b27-bfc8-fc359bb726f3,ENGI11.SA,2026-06-11,45.720001,47.080002,45.419998,47.080002,1715800,2026-06-12,2026-06-12 22:24:01.881502


In [17]:
def clear_silver():

    conn = duckdb.connect(DB_PATH)

    conn.execute("DELETE FROM silver.stock_fundamentals")
    conn.execute("DELETE FROM silver.stock_price_history")
    conn.execute("DELETE FROM silver.rejected_records")
    conn.execute("DELETE FROM silver.data_quality_report")

    conn.close()

In [18]:
def transform_fundamentals_to_silver():

    conn = duckdb.connect(DB_PATH)

    valid_df = conn.sql("""

    SELECT DISTINCT

        ticker,
        company_name,
        sector,

        current_price,
        market_cap,
        enterprise_value,

        trailing_pe,
        price_to_book,

        dividend_yield,

        beta,

        ebitda,

        total_debt,

        revenue_growth,
        earnings_growth,

        return_on_equity,
        return_on_assets,

        snapshot_date

    FROM bronze.stock_fundamentals_raw

    WHERE ticker IS NOT NULL
      AND current_price > 0
      AND market_cap > 0
      AND snapshot_date IS NOT NULL

    """).df()

    conn.register("valid_df", valid_df)

    conn.execute("""

    INSERT INTO silver.stock_fundamentals

    SELECT *

    FROM valid_df

    """)

    conn.close()

    return len(valid_df)

In [19]:
def load_rejected_fundamentals():

    conn = duckdb.connect(DB_PATH)

    rejected = conn.sql("""

    SELECT

        ticker,

        CASE

            WHEN ticker IS NULL
                THEN 'TICKER_NULL'

            WHEN current_price IS NULL
                THEN 'PRICE_NULL'

            WHEN current_price <= 0
                THEN 'PRICE_INVALID'

            WHEN market_cap IS NULL
                THEN 'MARKET_CAP_NULL'

            WHEN market_cap <= 0
                THEN 'MARKET_CAP_INVALID'

            ELSE 'UNKNOWN'

        END AS error_reason,

        CURRENT_TIMESTAMP AS load_timestamp

    FROM bronze.stock_fundamentals_raw

    WHERE

        ticker IS NULL

        OR current_price IS NULL

        OR current_price <= 0

        OR market_cap IS NULL

        OR market_cap <= 0

    """).df()

    if len(rejected) > 0:

        conn.register("rejected_df", rejected)

        conn.execute("""

        INSERT INTO silver.rejected_records

        SELECT *

        FROM rejected_df

        """)

    conn.close()

    return len(rejected)

In [20]:
def transform_prices_to_silver():

    conn = duckdb.connect(DB_PATH)

    valid_prices = conn.sql("""

    SELECT DISTINCT

        ticker,

        trade_date,

        open,
        high,
        low,
        close,

        volume

    FROM bronze.stock_price_history_raw

    WHERE ticker IS NOT NULL

      AND trade_date IS NOT NULL

      AND close > 0

      AND volume >= 0

    """).df()

    conn.register("prices_df", valid_prices)

    conn.execute("""

    INSERT INTO silver.stock_price_history

    SELECT *

    FROM prices_df

    """)

    conn.close()

    return len(valid_prices)

In [21]:
def build_quality_report():

    conn = duckdb.connect(DB_PATH)

    total_raw = conn.sql("""
    SELECT COUNT(*)
    FROM bronze.stock_fundamentals_raw
    """).fetchone()[0]

    total_silver = conn.sql("""
    SELECT COUNT(*)
    FROM silver.stock_fundamentals
    """).fetchone()[0]

    rejected = total_raw - total_silver

    report = pd.DataFrame([{

        "execution_time": datetime.now(),

        "rule_name":
        "fundamentals_validation",

        "approved":
        total_silver,

        "rejected":
        rejected

    }])

    conn.register(
        "quality_report_df",
        report
    )

    conn.execute("""

    INSERT INTO silver.data_quality_report

    SELECT *

    FROM quality_report_df

    """)

    conn.close()

In [27]:
def run_silver():



    valid_fundamentals = (
        transform_fundamentals_to_silver()
    )

    rejected = (
        load_rejected_fundamentals()
    )

    valid_prices = (
        transform_prices_to_silver()
    )

    build_quality_report()

    print("\nSilver concluída.")

    print(
        f"Fundamentais válidos: "
        f"{valid_fundamentals}"
    )

    print(
        f"Registros rejeitados: "
        f"{rejected}"
    )

    print(
        f"Histórico válido: "
        f"{valid_prices}"
    )

In [23]:
run_silver()


Silver concluída.
Fundamentais válidos: 8
Registros rejeitados: 1
Histórico válido: 2249


In [24]:
conn = duckdb.connect(DB_PATH)

conn.sql("""

SELECT *

FROM silver.data_quality_report

""").df()

,execution_time,rule_name,approved,rejected
0,2026-06-12 22:26:40.116550,fundamentals_validation,8,2


In [25]:
conn.sql("""

SELECT *

FROM silver.rejected_records

""").df()

,ticker,error_reason,load_timestamp
0,CEBR3.SA,MARKET_CAP_NULL,2026-06-12 22:26:40.088


In [26]:
conn.sql("""

SELECT *

FROM silver.stock_fundamentals

LIMIT 10

""").df()

,ticker,company_name,sector,current_price,market_cap,enterprise_value,trailing_pe,price_to_book,dividend_yield,beta,ebitda,total_debt,revenue_growth,earnings_growth,return_on_equity,return_on_assets,snapshot_date
0,TAEE11.SA,TAESA UNT N2,Utilities,39.69,4.101961e+10,2.392825e+10,39.297028,1.717884,8.2500,-0.014,2.324641e+09,1.185530e+10,0.055,-0.032,0.20559,0.06456,2026-06-12
1,EGIE3.SA,ENGIE BRASILON NM,Utilities,35.21,4.022034e+10,6.653766e+10,15.648889,2.957084,5.3000,0.302,7.146892e+09,3.242130e+10,0.131,-0.022,0.20204,0.06404,2026-06-12
2,ENGI11.SA,ENERGISA UNT N2,Utilities,47.40,1.451782e+10,5.640542e+10,34.347828,0.922125,6.9500,0.125,8.119768e+09,4.821897e+10,0.070,-0.400,0.12091,0.04653,2026-06-12
3,CMIG4.SA,CEMIG PN N1,Utilities,10.73,3.069512e+10,4.914859e+10,6.274853,1.062587,11.7400,0.093,7.750017e+09,2.001137e+10,0.063,-0.058,0.17038,0.05927,2026-06-12
4,AXIA3.SA,AXIA ENERGIAON NM,Utilities,52.00,1.169494e+11,2.050288e+11,11.818182,1.218940,6.9800,0.234,1.337589e+10,7.838196e+10,0.221,NaN,0.07862,0.01973,2026-06-12
5,ALUP11.SA,ALUPAR UNT N2,Utilities,32.27,1.534942e+10,2.508938e+10,7.733046,1.145382,2.5800,0.175,2.895635e+09,1.457164e+10,0.026,-0.338,0.12679,0.05156,2026-06-12
6,EQTL3.SA,EQUATORIAL ON NM,Utilities,38.77,4.865310e+10,9.872817e+10,41.244682,1.897700,6.1000,0.039,1.188615e+10,5.805280e+10,0.120,-0.240,0.06936,0.04946,2026-06-12
7,NEOE3.SA,NEOENERGIA ON,Utilities,33.80,4.102635e+10,9.104866e+10,7.716895,1.125504,0.0433,0.284,1.425800e+10,5.901900e+10,0.028,0.731,0.14588,0.06113,2026-06-12


In [28]:
def build_valuation_metrics():

    conn = duckdb.connect(DB_PATH)

    conn.execute("""
    DELETE FROM gold.valuation_metrics
    """)

    conn.execute("""

    INSERT INTO gold.valuation_metrics

    SELECT

        ticker,

        trailing_pe AS pe_ratio,

        price_to_book AS pb_ratio,

        ROUND(
            enterprise_value /
            NULLIF(ebitda,0),
            2
        ) AS ev_ebitda,

        dividend_yield,

        snapshot_date

    FROM silver.stock_fundamentals

    """)

    conn.close()

    print("Valuation concluída.")

In [30]:
def build_risk_metrics():

    conn = duckdb.connect(DB_PATH)

    returns = conn.sql("""

    WITH base AS (

        SELECT

            ticker,

            trade_date,

            close,

            close /

            LAG(close)
            OVER(
                PARTITION BY ticker
                ORDER BY trade_date
            )

            - 1 AS daily_return

        FROM silver.stock_price_history

    )

    SELECT

        ticker,

        STDDEV(daily_return)
        AS volatility

    FROM base

    WHERE daily_return IS NOT NULL

    GROUP BY ticker

    """).df()

    conn.register("returns_df", returns)

    conn.execute("""
    DELETE FROM gold.risk_metrics
    """)

    conn.execute("""

    INSERT INTO gold.risk_metrics

    SELECT

        f.ticker,

        f.beta,

        ROUND(
            f.total_debt /
            NULLIF(f.ebitda,0),
            2
        ) AS debt_ebitda,

        r.volatility,

        NULL,

        f.snapshot_date

    FROM silver.stock_fundamentals f

    LEFT JOIN returns_df r

        ON f.ticker = r.ticker

    """)

    conn.close()

In [31]:
def build_performance_metrics():

    conn = duckdb.connect(DB_PATH)

    conn.execute("""
    DELETE FROM gold.performance_metrics
    """)

    price_df = conn.sql("""

    SELECT *

    FROM silver.stock_price_history

    ORDER BY ticker, trade_date

    """).df()

    rows = []

    for ticker, group in price_df.groupby("ticker"):

        group = group.sort_values("trade_date")

        try:

            last_price = group["close"].iloc[-1]

            rows.append({

                "ticker": ticker,

                "return_30d":

                    (
                        last_price /
                        group["close"].iloc[-30]
                        - 1
                    ) * 100,

                "return_90d":

                    (
                        last_price /
                        group["close"].iloc[-90]
                        - 1
                    ) * 100,

                "return_180d":

                    (
                        last_price /
                        group["close"].iloc[-180]
                        - 1
                    ) * 100,

                "return_365d":

                    (
                        last_price /
                        group["close"].iloc[0]
                        - 1
                    ) * 100

            })

        except:
            pass

    perf = pd.DataFrame(rows)

    perf["snapshot_date"] = datetime.now().date()

    conn.register("perf_df", perf)

    conn.execute("""

    INSERT INTO gold.performance_metrics

    SELECT *

    FROM perf_df

    """)

    conn.close()

    print("Performance concluída.")

In [32]:
def build_data_mart():

    conn = duckdb.connect(DB_PATH)

    conn.execute("""
    DELETE FROM mart.dashboard_dataset
    """)

    conn.execute("""

    INSERT INTO mart.dashboard_dataset

    SELECT

        f.ticker,

        f.company_name,

        f.sector,

        f.current_price,

        v.pe_ratio,

        v.pb_ratio,

        v.ev_ebitda,

        v.dividend_yield,

        r.beta,

        r.debt_ebitda,

        r.volatility,

        r.max_drawdown,

        p.return_30d,

        p.return_90d,

        p.return_180d,

        p.return_365d,

        f.snapshot_date

    FROM silver.stock_fundamentals f

    LEFT JOIN gold.valuation_metrics v

        ON f.ticker = v.ticker

    LEFT JOIN gold.risk_metrics r

        ON f.ticker = r.ticker

    LEFT JOIN gold.performance_metrics p

        ON f.ticker = p.ticker

    """)

    conn.close()

    print("Data Mart criada.")

In [33]:
def run_gold():

    build_valuation_metrics()

    build_risk_metrics()

    build_performance_metrics()

    build_data_mart()

    print("\nGold concluída.")

In [34]:
run_gold()

Valuation concluída.
Performance concluída.
Data Mart criada.

Gold concluída.


In [35]:
conn = duckdb.connect(DB_PATH)

conn.sql("""
SELECT *
FROM mart.dashboard_dataset
""").df()

,ticker,company_name,sector,current_price,pe_ratio,pb_ratio,ev_ebitda,dividend_yield,beta,debt_ebitda,volatility,max_drawdown,return_30d,return_90d,return_180d,return_365d,snapshot_date
0,TAEE11.SA,TAESA UNT N2,Utilities,39.69,39.297028,1.717884,10.29,8.2500,-0.014,5.10,0.013281,NaN,-5.378078,-1.246818,17.247909,26.779631,2026-06-12
1,EGIE3.SA,ENGIE BRASILON NM,Utilities,35.21,15.648889,2.957084,9.31,5.3000,0.302,4.54,0.016603,NaN,1.121529,8.959449,20.219569,27.034091,2026-06-12
2,ENGI11.SA,ENERGISA UNT N2,Utilities,47.40,34.347828,0.922125,6.95,6.9500,0.125,5.94,0.016317,NaN,-10.261261,-7.077038,4.858320,15.636593,2026-06-12
3,CMIG4.SA,CEMIG PN N1,Utilities,10.73,6.274853,1.062587,6.34,11.7400,0.093,2.58,0.014512,NaN,-12.790069,-2.928305,4.149312,13.880266,2026-06-12
4,AXIA3.SA,AXIA ENERGIAON NM,Utilities,52.00,11.818182,1.218940,15.33,6.9800,0.234,5.86,0.023480,NaN,-16.196615,-4.359020,6.070095,34.805157,2026-06-12
5,ALUP11.SA,ALUPAR UNT N2,Utilities,32.27,7.733046,1.145382,8.66,2.5800,0.175,5.03,0.012711,NaN,-5.857145,-4.253068,6.198880,10.323961,2026-06-12
6,EQTL3.SA,EQUATORIAL ON NM,Utilities,38.77,41.244682,1.897700,8.31,6.1000,0.039,4.88,0.015527,NaN,-8.388467,-5.254149,8.699200,10.663095,2026-06-12
7,NEOE3.SA,NEOENERGIA ON,Utilities,33.80,7.716895,1.125504,6.39,0.0433,0.284,4.14,0.011593,NaN,0.000000,4.224480,21.960241,43.425320,2026-06-12


In [36]:
def create_analytics_views():

    conn = duckdb.connect(DB_PATH)

    conn.execute("""
    CREATE OR REPLACE VIEW mart.vw_valuation_ranking AS

    SELECT

        ticker,

        pe_ratio,

        pb_ratio,

        ev_ebitda,

        dividend_yield

    FROM mart.dashboard_dataset
    """)

    conn.execute("""
    CREATE OR REPLACE VIEW mart.vw_risk_ranking AS

    SELECT

        ticker,

        beta,

        debt_ebitda,

        volatility

    FROM mart.dashboard_dataset
    """)

    conn.execute("""
    CREATE OR REPLACE VIEW mart.vw_returns_ranking AS

    SELECT

        ticker,

        return_30d,

        return_90d,

        return_180d,

        return_365d

    FROM mart.dashboard_dataset
    """)

    conn.close()

    print("Views criadas.")

In [37]:
create_analytics_views()

Views criadas.


In [38]:
%%writefile app.py
import streamlit as st
import duckdb
import pandas as pd
import plotly.express as px

st.set_page_config(
    page_title="B3 Energy Analytics",
    layout="wide"
)

conn = duckdb.connect("energy_market.duckdb")

df = conn.sql("""
SELECT *
FROM mart.dashboard_dataset
""").df()

audit = conn.sql("""
SELECT *
FROM bronze.pipeline_audit
""").df()

quality = conn.sql("""
SELECT *
FROM silver.data_quality_report
""").df()

st.title("⚡ B3 Energy Analytics Platform")

page = st.sidebar.radio(
    "Navegação",
    [
        "Overview",
        "Valuation",
        "Risk",
        "Performance",
        "Governança"
    ]
)

if page == "Overview":

    c1,c2,c3,c4 = st.columns(4)

    c1.metric(
        "Ativos",
        len(df)
    )

    c2.metric(
        "Preço Médio",
        round(df["current_price"].mean(),2)
    )

    c3.metric(
        "DY Médio",
        round(df["dividend_yield"].mean(),2)
    )

    c4.metric(
        "Beta Médio",
        round(df["beta"].mean(),2)
    )

    fig = px.scatter(
        df,
        x="beta",
        y="ev_ebitda",
        size="current_price",
        hover_name="ticker"
    )

    st.plotly_chart(
        fig,
        use_container_width=True
    )

elif page == "Valuation":

    fig = px.bar(
        df.sort_values(
            "pe_ratio"
        ),
        x="ticker",
        y="pe_ratio"
    )

    st.plotly_chart(
        fig,
        use_container_width=True
    )

    st.dataframe(
        df[
            [
                "ticker",
                "pe_ratio",
                "pb_ratio",
                "ev_ebitda",
                "dividend_yield"
            ]
        ]
    )

elif page == "Risk":

    fig = px.bar(
        df,
        x="ticker",
        y="beta"
    )

    st.plotly_chart(
        fig,
        use_container_width=True
    )

    st.dataframe(
        df[
            [
                "ticker",
                "beta",
                "debt_ebitda",
                "volatility"
            ]
        ]
    )

elif page == "Performance":

    fig = px.bar(
        df,
        x="ticker",
        y="return_365d"
    )

    st.plotly_chart(
        fig,
        use_container_width=True
    )

    st.dataframe(
        df[
            [
                "ticker",
                "return_30d",
                "return_90d",
                "return_180d",
                "return_365d"
            ]
        ]
    )

else:

    st.subheader("Pipeline Audit")

    st.dataframe(audit)

    st.subheader("Data Quality")

    st.dataframe(quality)

Writing app.py


In [39]:
!pip install streamlit pyngrok

In [42]:
!npm install -g localtunnel

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋
added 22 packages in 2s
⠋
⠋3 packages are looking for funding
⠋  run `npm fund` for details
⠋

In [43]:
!streamlit run app.py &>/content/logs.txt &

In [47]:
!lt --port 8501

your url is: https://pretty-guests-joke.loca.lt
^C


In [48]:
# ============================================================
# ENTREGA DE VALOR
# CAMADA DE CONSUMO (SERVING LAYER)
# ============================================================
#
# Objetivo:
# Disponibilizar os dados processados na camada Gold através
# de uma tabela consolidada (Data Mart), permitindo consultas,
# exportações e consumo por dashboards ou APIs.
#
# Fonte:
# mart.dashboard_dataset
#
# ============================================================

import duckdb
import pandas as pd

# Conexão com o banco
conn = duckdb.connect("energy_market.duckdb")

In [49]:
# ============================================================
# CARREGAMENTO DOS DADOS PARA CONSUMO
# ============================================================

dashboard_df = conn.sql("""

SELECT *

FROM mart.dashboard_dataset

ORDER BY ticker

""").df()

print(f"Total de registros: {len(dashboard_df)}")

dashboard_df.head()

Total de registros: 8


,ticker,company_name,sector,current_price,pe_ratio,pb_ratio,ev_ebitda,dividend_yield,beta,debt_ebitda,volatility,max_drawdown,return_30d,return_90d,return_180d,return_365d,snapshot_date
0,ALUP11.SA,ALUPAR UNT N2,Utilities,32.27,7.733046,1.145382,8.66,2.58,0.175,5.03,0.012711,NaN,-5.857145,-4.253068,6.198880,10.323961,2026-06-12
1,AXIA3.SA,AXIA ENERGIAON NM,Utilities,52.00,11.818182,1.218940,15.33,6.98,0.234,5.86,0.023480,NaN,-16.196615,-4.359020,6.070095,34.805157,2026-06-12
2,CMIG4.SA,CEMIG PN N1,Utilities,10.73,6.274853,1.062587,6.34,11.74,0.093,2.58,0.014512,NaN,-12.790069,-2.928305,4.149312,13.880266,2026-06-12
3,EGIE3.SA,ENGIE BRASILON NM,Utilities,35.21,15.648889,2.957084,9.31,5.30,0.302,4.54,0.016603,NaN,1.121529,8.959449,20.219569,27.034091,2026-06-12
4,ENGI11.SA,ENERGISA UNT N2,Utilities,47.40,34.347828,0.922125,6.95,6.95,0.125,5.94,0.016317,NaN,-10.261261,-7.077038,4.858320,15.636593,2026-06-12


In [50]:
# ============================================================
# TOP EMPRESAS POR DIVIDEND YIELD
# ============================================================

top_dividend = conn.sql("""

SELECT

    ticker,
    dividend_yield

FROM mart.dashboard_dataset

ORDER BY dividend_yield DESC

""").df()

top_dividend

,ticker,dividend_yield
0,CMIG4.SA,11.7400
1,TAEE11.SA,8.2500
2,AXIA3.SA,6.9800
3,ENGI11.SA,6.9500
4,EQTL3.SA,6.1000
5,EGIE3.SA,5.3000
6,ALUP11.SA,2.5800
7,NEOE3.SA,0.0433


In [58]:
# ============================================================
# RANKING DE DIVIDEND YIELD
# ============================================================

import plotly.express as px

dividend_df = conn.sql("""

SELECT

    ticker,
    dividend_yield

FROM mart.dashboard_dataset

WHERE dividend_yield IS NOT NULL

ORDER BY dividend_yield DESC

""").df()

fig = px.bar(
    dividend_df,
    x="ticker",
    y="dividend_yield",
    title="Ranking de Dividend Yield",
    text_auto=".2%"
)

fig.update_layout(
    xaxis_title="Ativo",
    yaxis_title="Dividend Yield"
)

fig.show()

In [51]:
# ============================================================
# EMPRESAS COM MENOR P/L
# ============================================================

pl_ranking = conn.sql("""

SELECT

    ticker,
    pe_ratio

FROM mart.dashboard_dataset

WHERE pe_ratio IS NOT NULL

ORDER BY pe_ratio

""").df()

pl_ranking

,ticker,pe_ratio
0,CMIG4.SA,6.274853
1,NEOE3.SA,7.716895
2,ALUP11.SA,7.733046
3,AXIA3.SA,11.818182
4,EGIE3.SA,15.648889
5,ENGI11.SA,34.347828
6,TAEE11.SA,39.297028
7,EQTL3.SA,41.244682


In [59]:
# ============================================================
# COMPARAÇÃO DE P/L
# ============================================================

pl_df = conn.sql("""

SELECT

    ticker,
    pe_ratio

FROM mart.dashboard_dataset

WHERE pe_ratio IS NOT NULL

ORDER BY pe_ratio

""").df()

fig = px.bar(
    pl_df,
    x="ticker",
    y="pe_ratio",
    title="Comparação de P/L"
)

fig.update_layout(
    xaxis_title="Ativo",
    yaxis_title="P/L"
)

fig.show()

In [52]:
# ============================================================
# EMPRESAS COM MENOR EV/EBITDA
# ============================================================

ev_ebitda_ranking = conn.sql("""

SELECT

    ticker,
    ev_ebitda

FROM mart.dashboard_dataset

WHERE ev_ebitda IS NOT NULL

ORDER BY ev_ebitda

""").df()

ev_ebitda_ranking

,ticker,ev_ebitda
0,CMIG4.SA,6.34
1,NEOE3.SA,6.39
2,ENGI11.SA,6.95
3,EQTL3.SA,8.31
4,ALUP11.SA,8.66
5,EGIE3.SA,9.31
6,TAEE11.SA,10.29
7,AXIA3.SA,15.33


In [60]:
# ============================================================
# COMPARAÇÃO DE EV/EBITDA
# ============================================================

ev_df = conn.sql("""

SELECT

    ticker,
    ev_ebitda

FROM mart.dashboard_dataset

WHERE ev_ebitda IS NOT NULL

ORDER BY ev_ebitda

""").df()

fig = px.bar(
    ev_df,
    x="ticker",
    y="ev_ebitda",
    title="Comparação de EV/EBITDA"
)

fig.update_layout(
    xaxis_title="Ativo",
    yaxis_title="EV/EBITDA"
)

fig.show()

In [53]:
# ============================================================
# EMPRESAS COM MENOR BETA
# ============================================================

beta_ranking = conn.sql("""

SELECT

    ticker,
    beta

FROM mart.dashboard_dataset

WHERE beta IS NOT NULL

ORDER BY beta

""").df()

beta_ranking

,ticker,beta
0,TAEE11.SA,-0.014
1,EQTL3.SA,0.039
2,CMIG4.SA,0.093
3,ENGI11.SA,0.125
4,ALUP11.SA,0.175
5,AXIA3.SA,0.234
6,NEOE3.SA,0.284
7,EGIE3.SA,0.302


In [61]:
# ============================================================
# COMPARAÇÃO DE BETA
# ============================================================

beta_df = conn.sql("""

SELECT

    ticker,
    beta

FROM mart.dashboard_dataset

WHERE beta IS NOT NULL

ORDER BY beta

""").df()

fig = px.bar(
    beta_df,
    x="ticker",
    y="beta",
    title="Comparação de Beta"
)

fig.update_layout(
    xaxis_title="Ativo",
    yaxis_title="Beta"
)

fig.show()

In [54]:
# ============================================================
# RETORNO DOS ÚLTIMOS 12 MESES
# ============================================================

performance_ranking = conn.sql("""

SELECT

    ticker,
    return_365d

FROM mart.dashboard_dataset

WHERE return_365d IS NOT NULL

ORDER BY return_365d DESC

""").df()

performance_ranking

,ticker,return_365d
0,NEOE3.SA,43.425320
1,AXIA3.SA,34.805157
2,EGIE3.SA,27.034091
3,TAEE11.SA,26.779631
4,ENGI11.SA,15.636593
5,CMIG4.SA,13.880266
6,EQTL3.SA,10.663095
7,ALUP11.SA,10.323961


In [62]:
# ============================================================
# PERFORMANCE DOS ATIVOS
# ============================================================

perf_df = conn.sql("""

SELECT

    ticker,
    return_365d

FROM mart.dashboard_dataset

WHERE return_365d IS NOT NULL

ORDER BY return_365d DESC

""").df()

fig = px.bar(
    perf_df,
    x="ticker",
    y="return_365d",
    title="Retorno Acumulado - 12 Meses",
    text_auto=".2f"
)

fig.update_layout(
    xaxis_title="Ativo",
    yaxis_title="Retorno (%)"
)

fig.show()

In [63]:
# ============================================================
# MATRIZ RISCO X RETORNO
# ============================================================

risk_return = conn.sql("""

SELECT

    ticker,
    volatility,
    return_365d

FROM mart.dashboard_dataset

WHERE volatility IS NOT NULL
  AND return_365d IS NOT NULL

""").df()

fig = px.scatter(
    risk_return,
    x="volatility",
    y="return_365d",
    text="ticker",
    title="Risco x Retorno"
)

fig.update_traces(
    textposition="top center"
)

fig.update_layout(
    xaxis_title="Volatilidade",
    yaxis_title="Retorno 12 Meses (%)"
)

fig.show()

In [64]:
# ============================================================
# KPIs EXECUTIVOS
# ============================================================

dashboard_df = conn.sql("""

SELECT *

FROM mart.dashboard_dataset

""").df()

print("ATIVOS MONITORADOS:", len(dashboard_df))

print(
    "PREÇO MÉDIO:",
    round(dashboard_df["current_price"].mean(),2)
)

print(
    "DIVIDEND YIELD MÉDIO:",
    round(dashboard_df["dividend_yield"].mean(),4)
)

print(
    "BETA MÉDIO:",
    round(dashboard_df["beta"].mean(),2)
)

print(
    "VOLATILIDADE MÉDIA:",
    round(dashboard_df["volatility"].mean(),4)
)

ATIVOS MONITORADOS: 8
PREÇO MÉDIO: 36.23
DIVIDEND YIELD MÉDIO: 5.9929
BETA MÉDIO: 0.15
VOLATILIDADE MÉDIA: 0.0155


In [65]:
# ============================================================
# DATA QUALITY
# ============================================================

quality_df = conn.sql("""

SELECT *

FROM silver.data_quality_report

""").df()

fig = px.bar(
    quality_df,
    x="rule_name",
    y=["approved", "rejected"],
    barmode="group",
    title="Relatório de Qualidade dos Dados"
)

fig.show()

In [55]:
# ============================================================
# EXPORTAÇÃO DOS DADOS PARA CONSUMO EXTERNO
# ============================================================
#
# Permite utilização em:
# - Power BI
# - Excel
# - Tableau
# - Qlik
# - Outros sistemas analíticos
#
# ============================================================

dashboard_df.to_csv(
    "dashboard_dataset.csv",
    index=False
)

print("Arquivo dashboard_dataset.csv gerado com sucesso.")

Arquivo dashboard_dataset.csv gerado com sucesso.


In [56]:
# ============================================================
# VIEWS ANALÍTICAS PARA CONSUMO
# ============================================================

conn.execute("""

CREATE OR REPLACE VIEW mart.vw_top_dividend_yield AS

SELECT

    ticker,
    dividend_yield

FROM mart.dashboard_dataset

ORDER BY dividend_yield DESC

""")

conn.execute("""

CREATE OR REPLACE VIEW mart.vw_risk_ranking AS

SELECT

    ticker,
    beta,
    debt_ebitda,
    volatility

FROM mart.dashboard_dataset

ORDER BY beta

""")

conn.execute("""

CREATE OR REPLACE VIEW mart.vw_performance_ranking AS

SELECT

    ticker,
    return_30d,
    return_90d,
    return_180d,
    return_365d

FROM mart.dashboard_dataset

ORDER BY return_365d DESC

""")

print("Views analíticas criadas com sucesso.")

Views analíticas criadas com sucesso.


In [57]:
# ============================================================
# EXEMPLO DE CONSUMO DAS VIEWS
# ============================================================

conn.sql("""

SELECT *

FROM mart.vw_top_dividend_yield

""").df()

,ticker,dividend_yield
0,CMIG4.SA,11.7400
1,TAEE11.SA,8.2500
2,AXIA3.SA,6.9800
3,ENGI11.SA,6.9500
4,EQTL3.SA,6.1000
5,EGIE3.SA,5.3000
6,ALUP11.SA,2.5800
7,NEOE3.SA,0.0433


Texto para o Relatório

Entrega de Valor – Disponibilização dos Dados para Consumo

Após o processamento nas camadas Bronze, Silver e Gold, os dados são disponibilizados através da camada Mart, responsável por consolidar indicadores financeiros, métricas de risco e desempenho dos ativos monitorados. Essa camada permite consultas estruturadas em SQL, exportação para arquivos CSV, integração com ferramentas de Business Intelligence e exposição via API. Dessa forma, os dados deixam de ser apenas registros processados e passam a apoiar efetivamente análises de valuation, monitoramento de risco e tomada de decisão sobre empresas do setor elétrico da B3.
